In [1]:
# Install PySCF and GPU4PySCF (CUDA 12.x version for Colab's T4 GPU)
# cuTENSOR and CuPy are also installed for maximum tensor contraction acceleration.
!pip install pyscf gpu4pyscf-cuda12x cupy-cuda12x pyscf-dispersion geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.0/386.0 kB 1.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 MB 22.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 MB 661.2 kB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 703.2 kB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 84.6 MB/s eta 0:00:00:00:010:01
  Created wheel for geometric: filename=geometric-1.1-py3-none-any.whl size=402087 sha256=bbfd7cbccc081222c4085feba4141cdb9818a12b0984bc9134549c5c4619feaf
  Stored in directory: /root/.cache/pip/wheels/df/1e/9a/763451b0dfd8db47fc02239e33cdf138cbebdbea352bb0871b
Successfully built geometric


**Only use the next cell in case multiple GPUs are used**

In [2]:
# Must be set BEFORE any gpu4pyscf imports
import gpu4pyscf.__config__ as gpu4pyscf_config
gpu4pyscf_config.num_devices = 2

# Verify both GPUs
import cupy as cp
for i in range(2):
    with cp.cuda.Device(i):
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU {i}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

/usr/local/lib/python3.12/dist-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


GPU 0: 15.5 GB free / 15.6 GB total
GPU 1: 15.5 GB free / 15.6 GB total


In [3]:
import os
# Please use an xTB/other tool optimised file. Example naming is xtbopt.xyz. Please refrain from using other file formats 
# Copy the file path uploaded as a dataset in the input tab of Kaggle or system path if run locally 
xyz_filename = "/kaggle/input/datasets/pcsciprav/xtbopt-example/xtbopt.xyz"

print(f"Successfully loaded: {xyz_filename}")

Successfully loaded: /kaggle/input/datasets/pcsciprav/xtbopt-example/xtbopt.xyz


In [4]:
import cupy as cp

# Verify both GPUs are visible
n_gpus = cp.cuda.runtime.getDeviceCount()
print(f"GPUs available: {n_gpus}")
for i in range(n_gpus):
    with cp.cuda.Device(i):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  GPU {i}: {props['name'].decode()}, {props['totalGlobalMem'] / 1e9:.1f} GB")

GPUs available: 2
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


In [5]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # Enable both T4s before GPU objects

import pyscf
from pyscf import gto
from gpu4pyscf.dft import rks
from pyscf.geomopt.geometric_solver import optimize
import cupy as cp

# 2. Initialize the Molecule from the uploaded xyz file
with open(xyz_filename, 'r') as f:
    lines = f.readlines()

# XYZ format: line 0 = atom count, line 1 = comment, line 2+ = atoms
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]  # skip the xtb comment line
xyz_contents = ''.join(atom_lines)

mol = gto.M(
    atom=xyz_contents,
    basis = 'def2-TZVP', # Use def2-TZVP for higher accuracy, def2-SVP is lighter and causes less OOM issues, use crenbl with ecp.
    #ecp = 'crenbl', 
    charge=0,
    spin=0, # Here spin is 2S not 2S+1
    verbose=4,
    unit='angstrom'
)
mol.max_memory = 10000
print(f"\nMolecule initialized: {mol.natm} atoms, {mol.nao} atomic orbitals.")

# 3. Configure the GPU-Accelerated DFT Calculation
mf_pbe = rks.RKS(mol)
mf_pbe.xc = 'pbe-d3bj'
mf_pbe.kernel()
mf = rks.RKS(mol)
mf.xc = 'r2scan-d3bj'
dm = mf_pbe.make_rdm1()
mf.kernel(dm)
mf = mf.density_fit()

# Free all VRAM before optimization
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

# Lower memory usage per SCF cycle
mf.max_memory = 10000
mf.max_cycle = 400
mf.conv_tol = 1e-8
mf.conv_tol_grad = 1e-5
mf.diis_space = 12
mf.level_shift = 0.3
mf.damp = 0.4

# DO NOT use a fresh minao guess here
# mf.init_guess = 'minao'

# 4. RUN GEOMETRY OPTIMIZATION
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated Geometry Optimization...")

def save_callback(envs):
    mol_envs = envs['mol']
    mol_envs.tofile('/kaggle/working/dft_optimized_checkpoint.xyz')

mol_eq = optimize(
    mf,
    maxsteps=350,
    trust=0.03,
    tmax=0.03,
    callback=save_callback,
    assert_convergence=False
)

# 5. Save the results
output_file = 'dft_optimized.xyz'
mol_eq.tofile(output_file)

print("\n" + "="*50)
print(f"Optimization complete! New coordinates saved to: {output_file}")
print("="*50)

System: uname_result(system='Linux', node='f358eef02b07', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Sat Apr  4 07:56:54 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 18
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.

geometric-optimize called with the following command line:
/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-c8667e4a-07d0-4b8a-a203-0bddfdd56f46.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              ********    


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.359552  -0.077263  -0.009343   -0.000000  0.000000  0.000000
   N   1.824048  -0.100217   0.071883    0.000000  0.000000  0.000000
   H   0.054536   0.562637   0.723211    0.000000  0.000000  0.000000
   H   0.054536  -1.005934   0.279926    0.000000  0.000000  0.000000
   H   2.129064   0.828454  -0.217386    0.000000  0.000000  0.000000
   H   2.129064  -0.740117  -0.660671    0.000000  0.000000  0.000000

WARN: Mole.unit (angstrom) is changed to Bohr

New geometry
   1 N      0.359551988608  -0.077262628178  -0.009342913386 AA    0.679454786012  -0.146005206920  -0.017655547504 Bohr
   2 N      1.824048011272  -0.100217371631   0.071882913353 AA    3.446951179361  -0.189383385306   0.135839019273 Bohr
   3 H      0.054536065150   0.562636707799   0.723211240505 AA    0.103058227045   1.063229285367   1.366671174761 Bohr
   4 H      0.054536029865  

Step    0 : Gradient = 8.219e-03/1.176e-02 (rms/max) Energy = -111.8422693094
Hessian Eigenvalues: 2.30000e-02 5.00000e-02 5.00000e-02 ... 4.44367e-01 4.44367e-01 4.44367e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.359069  -0.070683  -0.032692   -0.000483  0.006580 -0.023349
   N   1.824531  -0.106797   0.095232    0.000483 -0.006580  0.023349
   H   0.066007   0.544486   0.720606    0.011471 -0.018151 -0.002605
   H   0.066041  -0.989161   0.287162    0.011505  0.016773  0.007236
   H   2.117559   0.811681  -0.224622   -0.011505 -0.016773 -0.007236
   H   2.117593  -0.721966  -0.658066   -0.011471  0.018151  0.002605
New geometry
   1 N      0.359069135943  -0.070682634856  -0.032691523398 AA    0.678542326716  -0.133570821640  -0.061778025817 Bohr
   2 N      1.824530864064  -0.106797365101   0.095231523409 AA    3.447863638897  -0.201817770865   0.179961497668 Bohr
   3 H      0.066007435032   0.544485941272   0.720606292137 AA    0.124735974395   1.028929307680   1.361748535777 Bohr
   4 H      0.066040945193  -0.989160652559   0.287162076182 AA    0.1247992

...


Geometry optimization cycle 4
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.353047  -0.069234  -0.037869   -0.000372  0.000513 -0.001861
   N   1.830553  -0.108246   0.100409    0.000372 -0.000513  0.001861
   H   0.058560   0.541272   0.723189    0.002586  0.002640 -0.000741
   H   0.058630  -0.987839   0.290963    0.002605 -0.001903 -0.002052
   H   2.124970   0.810359  -0.228423   -0.002605  0.001903  0.002052
   H   2.125040  -0.718752  -0.660649   -0.002586 -0.002640  0.000741
New geometry
   1 N      0.353046892633  -0.069233658588  -0.037869080249 AA    0.667161936205  -0.130832653333  -0.071562190259 Bohr
   2 N      1.830553106733  -0.108246341293   0.100409079995 AA    3.459244028197  -0.204555939030   0.189745661611 Bohr
   3 H      0.058560342418   0.541272064198   0.723189482326 AA    0.110663008931   1.022855960213   1.366630057761 Bohr
   4 H      0.058630376601  -0.987839336482   0.290963246985 AA    0.1107953

Step    3 : Displace = 3.248e-03/3.822e-03 (rms/max) Trust = 3.000e-02 (=) Grad = 3.753e-04/6.095e-04 (rms/max) E (change) = -111.8433526892 (-1.840e-05) Quality = 0.992
Hessian Eigenvalues: 2.30000e-02 4.96174e-02 5.00000e-02 ... 4.44367e-01 4.44367e-01 4.85595e-01



Geometry optimization cycle 5
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.352732  -0.069191  -0.038037   -0.000315  0.000042 -0.000168
   N   1.830868  -0.108289   0.100577    0.000315 -0.000042  0.000168
   H   0.058291   0.541776   0.722519   -0.000270  0.000504 -0.000671
   H   0.058381  -0.987947   0.290088   -0.000249 -0.000107 -0.000875
   H   2.125219   0.810467  -0.227548    0.000249  0.000107  0.000875
   H   2.125309  -0.719256  -0.659979    0.000270 -0.000504  0.000671
New geometry
   1 N      0.352731723591  -0.069191397793  -0.038037082497 AA    0.666566353033  -0.130752792004  -0.071879668496 Bohr
   2 N      1.830868274742  -0.108288601975   0.100577081849 AA    3.459839609417  -0.204635800145   0.190063139102 Bohr
   3 H      0.058290661541   0.541775776946   0.722518502499 AA    0.110153385933   1.023807839351   1.365362089653 Bohr
   4 H      0.058381095939  -0.987946822514   0.290088050006 AA    0.1103242

Step    4 : Displace = 3.963e-04/4.658e-04 (rms/max) Trust = 3.000e-02 (=) Grad = 1.126e-04/1.710e-04 (rms/max) E (change) = -111.8433532266 (-5.374e-07) Quality = 0.850
Hessian Eigenvalues: 2.30000e-02 4.96174e-02 5.00000e-02 ... 4.44367e-01 4.44367e-01 4.85595e-01
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 8.650 seconds



Optimization complete! New coordinates saved to: dft_optimized.xyz


In [6]:
import os
for f in os.listdir('/kaggle/working'):
    print(f)
import glob
print(glob.glob('/kaggle/working/*.xyz'))
print(glob.glob('/kaggle/working/**/*.xyz', recursive=True))

nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz
dft_optimized.xyz
.virtual_documents
nvhpc_2024_243_Linux_x86_64_cuda_12.3
dft_optimized_checkpoint.xyz
['/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_checkpoint.xyz']
['/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_checkpoint.xyz', '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/NEB/examples/ESM_example/reference/Al001+H_FCP_vm05.xyz', '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/NEB/examples/ESM_example/reference/Al001+H_bc3.xyz', '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/NEB/examples/ESM_example/reference/Al001+H_GCSCF_vm05.xyz', '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/NEB/examples/ESM_example/reference/Al001+H_bc3_n215.xyz', '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/NEB/examples/example01/reference/H2+H.xyz', '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/NEB/exampl

In [7]:
# Cell 4: Vibrational Frequencies and Thermochemistry
from gpu4pyscf.dft import rks
from gpu4pyscf.hessian import rks as gpu_hessian
from pyscf.hessian import thermo
import cupy as cp
import numpy as np
from pyscf import gto
import numpy as np

with open('/kaggle/working/dft_optimized.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-TZVP',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 16000

print("1. Setting up new SCF for the optimized geometry...")
mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'r2scan-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
e_opt = mf_eq.kernel()

print("\n2. Calculating analytical Hessian (Frequencies) on the GPU...")
hess = gpu_hessian.Hessian(mf_eq)
hess_matrix = hess.kernel()
# === AUTO-FIX IMAGINARY MODES LOOP ===

max_iter = 5
step = 0.1  # distortion magnitude (Å)

for i in range(max_iter):
    print("\n" + "="*40)
    print(f"Iteration {i+1}")
    print("="*40)

    # --- Compute Hessian ---
    hess = gpu_hessian.Hessian(mf_eq)
    hess_matrix = hess.kernel()

    # --- Reshape to 3N x 3N ---
    hess_2d = hess_matrix.reshape(3*n_atoms, 3*n_atoms)

    # --- Eigen decomposition ---
    eigvals, eigvecs = np.linalg.eigh(hess_2d)

    # --- Find REAL negative modes (ignore tiny noise) ---
    neg_indices = np.where(eigvals < -1e-3)[0]

    print(f"Negative eigenvalues: {len(neg_indices)}")
    print("Smallest eigenvalues:", eigvals[:6])

    # --- Stop if clean ---
    if len(neg_indices) == 0:
        print("✅ Reached true minimum (no imaginary modes)")
        break

    # --- Pick most negative mode ---
    mode_idx = neg_indices[0]
    mode = eigvecs[:, mode_idx].reshape(n_atoms, 3)

    coords = mol_eq.atom_coords()

    # --- Try BOTH directions ---
    best_energy = None
    best_coords = None

    for direction in [+1, -1]:
        trial_coords = coords + direction * step * mode

        mol_trial = mol_eq.copy()
        mol_trial.set_geom_(trial_coords, unit='angstrom')

        mf_trial = rks.RKS(mol_trial)
        mf_trial.xc = 'r2scan-d3bj'
        mf_trial = mf_trial.density_fit()
        mf_trial.conv_tol = 1e-9

        e_trial = mf_trial.kernel()

        print(f"  Direction {direction:+} Energy: {e_trial:.6f}")

        if best_energy is None or e_trial < best_energy:
            best_energy = e_trial
            best_coords = trial_coords

    # --- Update geometry to best ---
    mol_eq.set_geom_(best_coords, unit='angstrom')

    # --- Re-optimise ---
    print("Re-optimising...")
    mf_eq = rks.RKS(mol_eq)
    mf_eq.xc = 'r2scan-d3bj'
    mf_eq = mf_eq.density_fit()
    mf_eq.conv_tol = 1e-9
    mf_eq.kernel()

print("\nLoop complete.")
print("\n3. Calculating Thermochemistry (298.15 K, 1 atm)...")
freq_info = thermo.harmonic_analysis(mol_eq, hess_matrix)
thermo_data = thermo.thermo(mf_eq, freq_info['freq_au'], 298.15, 101325.0)

print("\n--- Thermochemistry Summary ---")
thermo.dump_thermo(mol_eq, thermo_data)

# MEMORY MANAGEMENT & BACKUP
print("\n4. Saving data to disk and purging VRAM...")

# Step A: Save the massive Hessian matrix to a file so we don't lose it
np.save('hessian_matrix.npy', np.asarray(hess_matrix))
print("   -> Hessian securely saved to 'hessian_matrix.npy'")

# Step B: Explicitly delete the giant objects from Python's memory
del hess
del freq_info
del thermo_data

# Step C: Force the GPU to empty the trash now that Python let go of the variables
cp.get_default_memory_pool().free_all_blocks()
print("   -> GPU VRAM successfully flushed!")

print("\n" + "="*50)
print("FREQUENCY & THERMO CALCULATION COMPLETE")
print("="*50)

System: uname_result(system='Linux', node='f358eef02b07', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Sat Apr  4 07:57:20 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 18
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.

In [8]:
import numpy as np
import cupy as cp

print("="*50)
print("HESSIAN / EIGENVALUE SUMMARY")
print("="*50)

H = hess_matrix
if isinstance(H, cp.ndarray):
    H = cp.asnumpy(H)

# block Hessian (natm,natm,3,3) -> full matrix
natm = H.shape[0]
Hfull = H.transpose(0,2,1,3).reshape(3*natm, 3*natm)

w = np.linalg.eigvalsh(Hfull)

print("Shape of full Hessian:", Hfull.shape)
print("Smallest 20 eigenvalues:")
print(w[:20])
print("Largest 10 eigenvalues:")
print(w[-10:])

tol = 1e-8
nneg = np.sum(w < -tol)
nzero = np.sum(np.abs(w) <= tol)
npos = np.sum(w > tol)

print("Negative eigenvalues:", nneg)
print("Near-zero eigenvalues:", nzero)
print("Positive eigenvalues:", npos)
print("="*50)
print("SOFTEST MODE ATOMIC PARTICIPATION")
print("="*50)

eigvals, eigvecs = np.linalg.eigh(Hfull)

for mode in range(min(5, len(eigvals))):
    vec = eigvecs[:, mode].reshape(natm, 3)
    atom_amp = np.linalg.norm(vec, axis=1)
    top = np.argsort(atom_amp)[::-1][:10]
    print(f"\nMode {mode}  eigenvalue = {eigvals[mode]: .8f}")
    for i in top:
        print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  amplitude = {atom_amp[i]:.6f}")
del hess_matrix

HESSIAN / EIGENVALUE SUMMARY
Shape of full Hessian: (18, 18)
Smallest 20 eigenvalues:
[-1.08010769e-01 -4.38534809e-03 -1.57414947e-03 -1.43547252e-03
 -8.86350069e-04 -2.48192861e-04 -2.42553654e-04 -1.62596288e-04
 -1.07161298e-04  2.22975650e-05  5.73497716e-05  6.97157765e-05
  1.26052139e-04  1.46414347e-04  1.96024266e-04  2.52688663e-04
  2.65885925e-04  4.20696692e-03]
Largest 10 eigenvalues:
[-1.07161298e-04  2.22975650e-05  5.73497716e-05  6.97157765e-05
  1.26052139e-04  1.46414347e-04  1.96024266e-04  2.52688663e-04
  2.65885925e-04  4.20696692e-03]
Negative eigenvalues: 9
Near-zero eigenvalues: 0
Positive eigenvalues: 9
SOFTEST MODE ATOMIC PARTICIPATION

Mode 0  eigenvalue = -0.10801077
Atom  0 N   amplitude = 0.999999
Atom  2 H   amplitude = 0.001136
Atom  3 H   amplitude = 0.000890
Atom  1 N   amplitude = 0.000652
Atom  5 H   amplitude = 0.000125
Atom  4 H   amplitude = 0.000083

Mode 1  eigenvalue = -0.00438535
Atom  0 N   amplitude = 0.999138
Atom  3 H   amplitude = 0.

In [9]:
# Cell 5: Excited States (TD-DFT) for UV-Vis
from gpu4pyscf.tdscf import rks as gpu_tdscf
import cupy as cp
from pyscf import gto
from gpu4pyscf.dft import rks

with open('dft_optimized_checkpoint.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-TZVP',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 10000

mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'r2scan-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
mf_eq.kernel()
print("Cleaning up GPU VRAM from previous calculations...")
# ==========================================
# CRITICAL VRAM CLEANUP
# Forces the GPU to release cached memory from the Hessian calc
# ==========================================
cp.get_default_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated TD-DFT...")
print("Calculating the first 5 singlet excited states...\n")

# Initialize Time-Dependent DFT using our optimized molecule's SCF object (mf_eq)
td = gpu_tdscf.TDDFT(mf_eq)
td.nstates = 5       # Number of excited states to calculate
td.singlet = True    # Look for singlet-singlet transitions

# ==========================================
# VRAM OPTIMIZATION FOR TD-DFT
# Restricts the Davidson solver's maximum memory footprint
# ==========================================
td.max_space = 15

# Run the calculation
td.kernel()

print("\n" + "="*50)
print("TD-DFT EXCITATION ENERGIES:")
print("="*50)
# This will print the transition energies (in eV), wavelengths (in nm), and oscillator strengths (intensity)
td.analyze()

System: uname_result(system='Linux', node='f358eef02b07', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Sat Apr  4 07:59:45 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 18
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.

<class 'gpu4pyscf.tdscf.rks.TDDFT'> does not have attributes  max_space


Excited State energies (eV)
[5.84671226 6.62083915 7.72096813 8.56649427 9.09276367]

TD-DFT EXCITATION ENERGIES:

** Singlet excitation energies and oscillator strengths **
Excited State   1:      5.84671 eV    212.06 nm  f=0.0000
       9 -> 10       -0.70683
Excited State   2:      6.62084 eV    187.26 nm  f=0.0259
       9 -> 11        0.70382
Excited State   3:      7.72097 eV    160.58 nm  f=0.0147
       9 -> 12       -0.70639
Excited State   4:      8.56649 eV    144.73 nm  f=0.0000
       9 -> 13       -0.70605
Excited State   5:      9.09276 eV    136.35 nm  f=0.0992
       9 -> 14       -0.69942

** Transition electric dipole moments (AU) **
state          X           Y           Z        Dip. S.      Osc.
  1         0.0000      0.0000     -0.0000      0.0000      0.0000
  2        -0.1512      0.1006     -0.3560      0.1597      0.0259
  3        -0.0000     -0.2679     -0.0757      0.0775      0.0147
  4        -0.0000      0.0000      0.0000      0.0000      0.0000
  5  

In [10]:
print("="*50)
print("TDDFT SUMMARY")
print("="*50)

print("Excitation energies (eV):")
print(cp.asnumpy(td.e) * 27.211386245988 if hasattr(td.e, "get") or "cupy" in str(type(td.e)).lower() else td.e * 27.211386245988)

print("\nOscillator strengths:")
try:
    print(td.oscillator_strength())
except Exception as e:
    print("Could not get oscillator strengths directly:", e)

TDDFT SUMMARY
Excitation energies (eV):
[5.84671231 6.62083921 7.7209682  8.56649434 9.09276375]

Oscillator strengths:
[1.08781628e-17 2.59100786e-02 1.46567850e-02 4.29690544e-19
 9.91987885e-02]


In [11]:
import cupy as cp
from pyscf.scf import hf

print("="*50)
print("1. MOLECULAR DIPOLE MOMENT")
print("="*50)
dipole = mf_eq.dip_moment()

print("\n" + "="*50)
print("2. MULLIKEN ATOMIC CHARGES")
print("="*50)

dm = mf_eq.make_rdm1()
s = mol_eq.intor('int1e_ovlp')

# Convert GPU array to CPU NumPy array
dm = cp.asnumpy(dm)

mulliken_pop, mulliken_chg = hf.mulliken_pop(mol_eq, dm, s=s)

print("\nAtom | Charge")
print("-" * 20)
for i, charge in enumerate(mulliken_chg):
    atom_symbol = mol_eq.atom_symbol(i)
    print(f"{i:2d} {atom_symbol:2s} | {charge: .4f}")

1. MOLECULAR DIPOLE MOMENT
Dipole moment(X, Y, Z, Debye):  0.00000, -0.00000,  0.00000

2. MULLIKEN ATOMIC CHARGES
 ** Mulliken pop  **
pop of  0 N 1s            0.81905
pop of  0 N 2s            1.15218
pop of  0 N 3s            0.57149
pop of  0 N 4s            0.83392
pop of  0 N 5s            0.24034
pop of  0 N 2px           0.40072
pop of  0 N 2py           0.49110
pop of  0 N 2pz           0.59799
pop of  0 N 3px           0.45208
pop of  0 N 3py           0.50682
pop of  0 N 3pz           0.61510
pop of  0 N 4px           0.12519
pop of  0 N 4py           0.19281
pop of  0 N 4pz           0.37598
pop of  0 N 3dxy          0.00090
pop of  0 N 3dyz          0.00215
pop of  0 N 3dz^2         0.00119
pop of  0 N 3dxz          0.00129
pop of  0 N 3dx2-y2       0.00221
pop of  0 N 4dxy          0.00807
pop of  0 N 4dyz          0.00619
pop of  0 N 4dz^2         0.00529
pop of  0 N 4dxz          0.01306
pop of  0 N 4dx2-y2       0.00819
pop of  0 N 4f-3          0.00053
pop of  0 N 4f

In [12]:
print("="*50)
print("ATOM INDEX MAP")
print("="*50)

for i in range(mol_eq.natm):
    x, y, z = mol_eq.atom_coord(i)
    print(f"{i:2d} {mol_eq.atom_symbol(i):2s}  {x: .6f} {y: .6f} {z: .6f}")

ATOM INDEX MAP
 0 N    0.666566 -0.130753 -0.071880
 1 N    3.459840 -0.204636  0.190063
 2 H    0.110153  1.023808  1.365362
 3 H    0.110324 -1.866949  0.548187
 4 H    4.016082  1.531560 -0.430003
 5 H    4.016253 -1.359196 -1.247179


In [13]:
import numpy as np

print("="*50)
print("CHARGE SUMMARY")
print("="*50)

charges = np.array(mulliken_chg)

print("Min charge:", charges.min())
print("Max charge:", charges.max())
print("Mean charge:", charges.mean())
print("Std dev:", charges.std())

print("\nTop 10 most positive atoms")
for i in np.argsort(charges)[::-1][:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

print("\nTop 10 most negative atoms")
for i in np.argsort(charges)[:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

CHARGE SUMMARY
Min charge: -0.42610288384058315
Max charge: 0.2130518581155798
Mean charge: 1.2582527612418441e-15
Std dev: 0.3013002378773471

Top 10 most positive atoms
Atom  2 H   q =  0.213052
Atom  5 H   q =  0.213052
Atom  3 H   q =  0.213051
Atom  4 H   q =  0.213051
Atom  0 N   q = -0.426103
Atom  1 N   q = -0.426103

Top 10 most negative atoms
Atom  1 N   q = -0.426103
Atom  0 N   q = -0.426103
Atom  4 H   q =  0.213051
Atom  3 H   q =  0.213051
Atom  5 H   q =  0.213052
Atom  2 H   q =  0.213052


In [14]:
# Cell 7: Save the optimized coordinates
output_xyz = 'dft_optimized_final.xyz'
mol_eq.tofile(output_xyz)
print(f"Saved to /kaggle/working/{output_xyz}")
print("Download it from the Output tab on the right panel.")

Saved to /kaggle/working/dft_optimized_final.xyz
Download it from the Output tab on the right panel.
